# AdvectNet Himawari checkpoint -> INSAT-3DR/MOSDAC fog test

This notebook does **not train**. It assumes the AdvectNet U-Net was already trained on Himawari-9 in the original notebook.

Inputs expected on Kaggle:

1. INSAT/MOSDAC fog zip containing files like `3RIMG_23JUN2026_0945_L2C_FOG_V01R00.h5`.
2. A trained Himawari checkpoint, for example `advectnet_unet_himawari.pth`.

Important: your original `isro-h-organised.ipynb` trains `unet` in memory but does not save it. After the training cell finishes there, run this once:

```python
torch.save({
    "model": unet.state_dict(),
    "patch": PATCH,
    "bt_min": BT_MIN,
    "bt_max": BT_MAX,
    "note": "AdvectNet U-Net trained on Himawari B13 normalized brightness temperature"
}, "advectnet_unet_himawari.pth")
```

Then upload that `.pth` as a Kaggle dataset together with, or separately from, the INSAT zip.

What this notebook tests:

- Direct inference: use consecutive 30-minute INSAT frames and synthesize the two missing intermediate frames at fractions $\alpha=1/3$ and $\alpha=2/3$.
- Proxy quantitative evaluation: because this zip has 30-minute cadence, there is no true 10-minute target. For measurable testing, it uses windows $(t,t+30,t+60,t+90)$, gives the model only $(t,t+90)$, predicts $t+30$ and $t+60$, then compares with the real middle frames. This is a harder temporal-transfer proxy, not the same physical task as Himawari 30min -> 10min.

## 1. Setup

In [ ]:
!pip install -q h5py scikit-image imageio

import os, re, io, zipfile, datetime as dt
from pathlib import Path

import h5py
import imageio.v2 as imageio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## 2. Configuration and Kaggle input discovery

In [ ]:
# If auto-discovery fails, set these manually, e.g.
# INSAT_SOURCE = "/kaggle/input/your-insat-dataset/3RIMG_23JUN2026_0945_L2C_FOG_V01R00.zip"
# INSAT_SOURCE = "/kaggle/input/your-insat-dataset"   # folder containing .h5 files also works
# WEIGHTS_PATH = "/kaggle/input/your-weights-dataset/advectnet_unet_himawari.pth"
INSAT_SOURCE = None
WEIGHTS_PATH = None

ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
OUT_DIR = Path("/kaggle/working/insat_advectnet_test")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIELD = "FOG"          # use FOG binary mask. FOG_INTENSITY is categorical and noisier for this model.
PATCH = 256
STRIDE = 256           # set 192 for smoother overlap, 256 for faster Kaggle runs
BATCH = 8              # reduce to 2/4 if CUDA OOM
MIN_VALID = 0.10       # skip tiles with less valid INSAT area than this
THR = 0.50             # fog probability threshold for binary metrics
MAX_EVAL_WINDOWS = None  # None = all proxy windows; set 3 for a quick smoke test
SAVE_FULL_NPZ = False    # True saves full stitched predictions; can be large

def all_input_files(roots=ROOTS):
    files = []
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file():
                files.append(p)
    return sorted(set(files), key=lambda p: str(p).lower())

all_files = all_input_files()
zip_hits = [p for p in all_files if p.suffix.lower() == ".zip"]
h5_hits = [p for p in all_files if p.suffix.lower() in [".h5", ".hdf5"]]
weight_hits = [p for p in all_files if p.suffix.lower() in [".pth", ".pt", ".ckpt"]]

fog_zips = [p for p in zip_hits if re.search(r"fog|l2c|3rimg", p.name, re.I)]
fog_h5 = [p for p in h5_hits if re.search(r"fog|l2c|3rimg", p.name, re.I)]

print("/kaggle/input file listing, first 120 files:")
for p in [x for x in all_files if str(x).startswith("/kaggle/input")][:120]:
    print("  ", p)
print("total files seen:", len(all_files))

print("candidate zips:")
for p in (fog_zips or zip_hits)[:30]:
    print("  ", p)
print("candidate h5 files:", len(fog_h5 or h5_hits))
for p in (fog_h5 or h5_hits)[:20]:
    print("  ", p)
print("candidate checkpoints:")
for p in weight_hits[:30]:
    print("  ", p)

if INSAT_SOURCE is None:
    if fog_zips:
        INSAT_SOURCE = str(fog_zips[0])
    elif zip_hits:
        INSAT_SOURCE = str(zip_hits[0])
    elif fog_h5:
        INSAT_SOURCE = str(fog_h5[0].parent)
    elif h5_hits:
        INSAT_SOURCE = str(h5_hits[0].parent)

if WEIGHTS_PATH is None:
    preferred = [p for p in weight_hits if re.search(r"advect|hima|unet", p.name, re.I)]
    WEIGHTS_PATH = str((preferred or weight_hits)[0]) if weight_hits else None

print("INSAT_SOURCE=", INSAT_SOURCE)
print("WEIGHTS_PATH=", WEIGHTS_PATH)
if not (INSAT_SOURCE and Path(INSAT_SOURCE).exists()):
    raise FileNotFoundError(
        "No INSAT source found under /kaggle/input. In Kaggle, click Add Data and attach the dataset containing "
        "the INSAT zip or extracted .h5 files. If it is already attached, copy one printed path above and set "
        "INSAT_SOURCE manually."
    )
if not (WEIGHTS_PATH and Path(WEIGHTS_PATH).exists()):
    raise FileNotFoundError(
        "No trained Himawari checkpoint found under /kaggle/input. Upload advectnet_unet_himawari.pth as a Kaggle dataset "
        "or set WEIGHTS_PATH manually."
    )

## 3. Read INSAT/MOSDAC HDF5 fog sequence

In [ ]:
def bstr(x):
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="ignore")
    if hasattr(x, "tolist"):
        x = x.tolist()
        if isinstance(x, bytes):
            return x.decode("utf-8", errors="ignore")
    return x

def parse_insat_time(name):
    m = re.search(r"3RIMG_(\d{2}[A-Z]{3}\d{4})_(\d{4})_", Path(name).name)
    if not m:
        raise ValueError(f"cannot parse INSAT timestamp from {name}")
    return dt.datetime.strptime(m.group(1) + m.group(2), "%d%b%Y%H%M")

def read_one_h5_bytes(blob, field=FIELD):
    with h5py.File(io.BytesIO(blob), "r") as f:
        return read_one_h5_open(f, field)

def read_one_h5_path(path, field=FIELD):
    with h5py.File(path, "r") as f:
        return read_one_h5_open(f, field)

def read_one_h5_open(f, field=FIELD):
    if field == "FOG":
        raw = f["FOG"][0]
        valid = raw != -128
        img = (raw == 1).astype(np.float32)
    elif field == "FOG_INTENSITY":
        raw = f["FOG_INTENSITY"][0]
        valid = f["FOG"][0] != -128
        # Nominal flags are 1..4. Unexpected packed values are clipped instead of trusted as linear intensity.
        img = np.where(valid, np.clip(raw.astype(np.float32), 0, 4) / 4.0, 0.0).astype(np.float32)
    else:
        raise ValueError(f"unknown FIELD={field}")
    img = np.where(valid, img, 0.0).astype(np.float32)
    meta = {k: bstr(v) for k, v in f.attrs.items()}
    return img, valid.astype(bool), meta

def load_insat_zip(zip_path):
    rows = []
    with zipfile.ZipFile(zip_path) as z:
        names = [n for n in z.namelist() if n.lower().endswith((".h5", ".hdf5"))]
        for name in names:
            rows.append((parse_insat_time(name), name))
        rows.sort()
        frames, masks, times, metas = [], [], [], []
        for t, name in rows:
            img, valid, meta = read_one_h5_bytes(z.read(name))
            frames.append(img)
            masks.append(valid)
            times.append(t)
            metas.append(meta)
            fog_px = int((img >= THR).sum())
            valid_px = int(valid.sum())
            print(f"{t:%Y-%m-%d %H:%M}  shape={img.shape}  valid={valid_px:,}  fog={fog_px:,}")
    return np.stack(frames), np.stack(masks), times, metas

def load_insat_h5_folder(folder):
    folder = Path(folder)
    files = sorted([p for p in folder.rglob("*.h5") if re.search(r"3RIMG_.*L2C_FOG", p.name)], key=lambda p: parse_insat_time(p.name))
    if not files:
        files = sorted(folder.rglob("*.h5"), key=lambda p: parse_insat_time(p.name))
    assert files, f"no .h5 files found under {folder}"
    frames, masks, times, metas = [], [], [], []
    for path in files:
        t = parse_insat_time(path.name)
        img, valid, meta = read_one_h5_path(path)
        frames.append(img)
        masks.append(valid)
        times.append(t)
        metas.append(meta)
        fog_px = int((img >= THR).sum())
        valid_px = int(valid.sum())
        print(f"{t:%Y-%m-%d %H:%M}  shape={img.shape}  valid={valid_px:,}  fog={fog_px:,}")
    return np.stack(frames), np.stack(masks), times, metas

def load_insat_source(source):
    source = Path(source)
    if source.is_file() and source.suffix.lower() == ".zip":
        return load_insat_zip(source)
    if source.is_file() and source.suffix.lower() in [".h5", ".hdf5"]:
        return load_insat_h5_folder(source.parent)
    if source.is_dir():
        zips = sorted(list(source.rglob("*L2C_FOG*.zip")) + list(source.rglob("*FOG*.zip")) + list(source.rglob("*.zip")))
        if zips:
            return load_insat_zip(zips[0])
        return load_insat_h5_folder(source)
    raise ValueError(f"unsupported INSAT_SOURCE: {source}")

frames, masks, times, metas = load_insat_source(INSAT_SOURCE)
N, H, W = frames.shape
print("loaded:", frames.shape, "from", times[0], "to", times[-1])
assert N >= 4, "Need at least 4 frames for proxy evaluation windows."

## 4. Model definition and trained checkpoint load

In [ ]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

# RAFT must match the training notebook: pretrained, frozen, used only for optical flow.
# If Kaggle internet is off and weights are not cached, turn internet on or add the torchvision RAFT cache as input.
raft = raft_small(weights=Raft_Small_Weights.DEFAULT).to(dev).eval()
for p in raft.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def raft_flow(a, b):
    a3 = (a * 2 - 1).repeat(1, 3, 1, 1)
    b3 = (b * 2 - 1).repeat(1, 3, 1, 1)
    return raft(a3, b3)[-1]

def backwarp(img, flow):
    B, C, h, w = img.shape
    yy, xx = torch.meshgrid(torch.arange(h, device=img.device),
                            torch.arange(w, device=img.device), indexing="ij")
    grid = torch.stack((xx, yy), 0).float()[None].repeat(B, 1, 1, 1) + flow
    gx = 2 * grid[:, 0] / max(w - 1, 1) - 1
    gy = 2 * grid[:, 1] / max(h - 1, 1) - 1
    return F.grid_sample(img, torch.stack((gx, gy), -1), mode="bilinear",
                         padding_mode="border", align_corners=True)

def inter_flows(F03, F30, alpha):
    Ft0 = -(1 - alpha) * alpha * F03 + alpha * alpha * F30
    Ft3 = (1 - alpha) ** 2 * F03 - alpha * (1 - alpha) * F30
    return Ft0, Ft3

def cbr(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, 1, 1), nn.GroupNorm(8, o), nn.GELU())

class UNet(nn.Module):
    def __init__(self, ic=8, b=32):
        super().__init__()
        self.e1 = nn.Sequential(cbr(ic, b), cbr(b, b))
        self.e2 = nn.Sequential(cbr(b, 2 * b), cbr(2 * b, 2 * b))
        self.e3 = nn.Sequential(cbr(2 * b, 4 * b), cbr(4 * b, 4 * b))
        self.pool = nn.MaxPool2d(2)
        self.d2 = nn.Sequential(cbr(4 * b + 2 * b, 2 * b), cbr(2 * b, 2 * b))
        self.d1 = nn.Sequential(cbr(2 * b + b, b), cbr(b, b))
        self.out = nn.Conv2d(b, 2, 3, 1, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        up = lambda z: F.interpolate(z, scale_factor=2, mode="bilinear", align_corners=False)
        d2 = self.d2(torch.cat([up(e3), e2], 1))
        d1 = self.d1(torch.cat([up(d2), e1], 1))
        o = self.out(d1)
        return torch.sigmoid(o[:, 0:1]), torch.tanh(o[:, 1:2])

unet = UNet().to(dev)

def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        for key in ["model", "unet", "state_dict", "model_state_dict"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                ckpt = ckpt[key]
                break
    if not isinstance(ckpt, dict):
        raise TypeError("checkpoint does not contain a state_dict-like object")
    out = {}
    for k, v in ckpt.items():
        nk = k
        for pref in ["module.", "unet.", "model."]:
            if nk.startswith(pref):
                nk = nk[len(pref):]
        out[nk] = v
    return out

ckpt = torch.load(WEIGHTS_PATH, map_location=dev)
state = extract_state_dict(ckpt)
missing, unexpected = unet.load_state_dict(state, strict=False)
print("missing keys:", missing)
print("unexpected keys:", unexpected)
assert len(missing) == 0, "checkpoint is not compatible with the AdvectNet U-Net architecture."
unet.eval()
print("checkpoint loaded")

## 5. Patchwise interpolation/stitching utilities

In [ ]:
def starts(n, patch=PATCH, stride=STRIDE):
    if n <= patch:
        return [0]
    vals = list(range(0, n - patch + 1, stride))
    if vals[-1] != n - patch:
        vals.append(n - patch)
    return vals

def pad_min(img, patch=PATCH, value=0):
    h, w = img.shape
    hp, wp = max(h, patch), max(w, patch)
    out = np.full((hp, wp), value, dtype=img.dtype)
    out[:h, :w] = img
    return out, (h, w)

@torch.no_grad()
def infer_batch(x0, x3, alpha):
    x0t = torch.tensor(x0, dtype=torch.float32, device=dev)[:, None]
    x3t = torch.tensor(x3, dtype=torch.float32, device=dev)[:, None]
    F03 = raft_flow(x0t, x3t)
    F30 = raft_flow(x3t, x0t)
    a = torch.full((len(x0), 1, 1, 1), float(alpha), device=dev)
    Ft0, Ft3 = inter_flows(F03, F30, a)
    w0 = backwarp(x0t, Ft0)
    w3 = backwarp(x3t, Ft3)
    blend = (1 - a) * x0t + a * x3t
    ach = a.expand(-1, 1, x0t.shape[-2], x0t.shape[-1])
    mask, res = unet(torch.cat([w0, w3, Ft0, Ft3, blend, ach], 1))
    out = (mask * w0 + (1 - mask) * w3 + res).clamp(0, 1)
    return out[:, 0].detach().cpu().numpy().astype(np.float32)

def interp_tiled(f0, f3, m0, m3, alpha, patch=PATCH, stride=STRIDE, batch=BATCH, min_valid=MIN_VALID):
    f0p, (h, w) = pad_min(f0, patch, 0)
    f3p, _ = pad_min(f3, patch, 0)
    m0p, _ = pad_min(m0.astype(bool), patch, False)
    m3p, _ = pad_min(m3.astype(bool), patch, False)
    hp, wp = f0p.shape
    out = np.zeros((hp, wp), np.float32)
    wgt = np.zeros((hp, wp), np.float32)
    win1 = np.hanning(patch).astype(np.float32)
    win = np.outer(win1, win1)
    win = np.maximum(win / max(win.max(), 1e-6), 0.05).astype(np.float32)

    jobs, coords = [], []
    for y in starts(hp, patch, stride):
        for x in starts(wp, patch, stride):
            vm = m0p[y:y+patch, x:x+patch] & m3p[y:y+patch, x:x+patch]
            if vm.mean() < min_valid:
                continue
            jobs.append((f0p[y:y+patch, x:x+patch], f3p[y:y+patch, x:x+patch]))
            coords.append((y, x))
            if len(jobs) == batch:
                pred = infer_batch(np.stack([j[0] for j in jobs]), np.stack([j[1] for j in jobs]), alpha)
                for p, (yy, xx) in zip(pred, coords):
                    out[yy:yy+patch, xx:xx+patch] += p * win
                    wgt[yy:yy+patch, xx:xx+patch] += win
                jobs, coords = [], []
    if jobs:
        pred = infer_batch(np.stack([j[0] for j in jobs]), np.stack([j[1] for j in jobs]), alpha)
        for p, (yy, xx) in zip(pred, coords):
            out[yy:yy+patch, xx:xx+patch] += p * win
            wgt[yy:yy+patch, xx:xx+patch] += win

    base = (1 - alpha) * f0p + alpha * f3p
    stitched = np.where(wgt > 0, out / np.maximum(wgt, 1e-6), base)
    return stitched[:h, :w].astype(np.float32)

print("tile grid:", len(starts(H)), "x", len(starts(W)), "=", len(starts(H)) * len(starts(W)), "tiles before validity filtering")

## 6. Proxy quantitative evaluation on INSAT frames

In [ ]:
def binary_scores(pred, gt, valid, thr=THR):
    valid = valid & np.isfinite(pred) & np.isfinite(gt)
    if valid.sum() == 0:
        return dict(mse=np.nan, psnr=np.nan, acc=np.nan, precision=np.nan, recall=np.nan, f1=np.nan, iou=np.nan)
    p = pred[valid]
    g = gt[valid]
    mse = float(np.mean((p - g) ** 2))
    psnr = float(10 * np.log10(1.0 / max(mse, 1e-12)))
    pb = p >= thr
    gb = g >= thr
    tp = int(np.logical_and(pb, gb).sum())
    tn = int(np.logical_and(~pb, ~gb).sum())
    fp = int(np.logical_and(pb, ~gb).sum())
    fn = int(np.logical_and(~pb, gb).sum())
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    iou = tp / max(tp + fp + fn, 1)
    return dict(mse=mse, psnr=psnr, acc=acc, precision=precision, recall=recall, f1=f1, iou=iou)

def eval_one(pred, base, gt, valid, tag, start_time, target_time, end_time):
    ms = binary_scores(pred, gt, valid)
    bs = binary_scores(base, gt, valid)
    row = {
        "window_start": start_time,
        "target_time": target_time,
        "window_end": end_time,
        "target": tag,
        "valid_pixels": int(valid.sum()),
        "fog_pixels_truth": int((gt[valid] >= THR).sum()),
    }
    for k, v in bs.items():
        row[f"baseline_{k}"] = v
    for k, v in ms.items():
        row[f"model_{k}"] = v
    return row

rows = []
limit = (N - 3) if MAX_EVAL_WINDOWS is None else min(MAX_EVAL_WINDOWS, N - 3)
for i in range(limit):
    f0, y1, y2, f3 = frames[i], frames[i+1], frames[i+2], frames[i+3]
    m0, m1, m2, m3 = masks[i], masks[i+1], masks[i+2], masks[i+3]
    valid_all = m0 & m1 & m2 & m3
    print(f"proxy window {i+1}/{limit}: {times[i]:%H:%M} -> {times[i+3]:%H:%M}")

    p1 = interp_tiled(f0, f3, m0, m3, 1/3)
    b1 = (2/3) * f0 + (1/3) * f3
    rows.append(eval_one(p1, b1, y1, valid_all, "alpha_1_3", times[i], times[i+1], times[i+3]))

    p2 = interp_tiled(f0, f3, m0, m3, 2/3)
    b2 = (1/3) * f0 + (2/3) * f3
    rows.append(eval_one(p2, b2, y2, valid_all, "alpha_2_3", times[i], times[i+2], times[i+3]))

metrics = pd.DataFrame(rows)
metrics_path = OUT_DIR / "insat_proxy_eval_metrics.csv"
metrics.to_csv(metrics_path, index=False)
print("saved", metrics_path)

display_cols = ["target", "baseline_f1", "model_f1", "baseline_iou", "model_iou", "baseline_mse", "model_mse", "valid_pixels", "fog_pixels_truth"]
display(metrics[display_cols].describe())
display(metrics[display_cols].head(10))

## 7. Visualize one proxy window

In [ ]:
VIS_I = 0
f0, y1, y2, f3 = frames[VIS_I], frames[VIS_I+1], frames[VIS_I+2], frames[VIS_I+3]
m0, m3 = masks[VIS_I], masks[VIS_I+3]
p1 = interp_tiled(f0, f3, m0, m3, 1/3)
p2 = interp_tiled(f0, f3, m0, m3, 2/3)
b1 = (2/3) * f0 + (1/3) * f3
b2 = (1/3) * f0 + (2/3) * f3

fig, ax = plt.subplots(2, 5, figsize=(18, 7))
items = [
    (f0, f"input {times[VIS_I]:%H:%M}"),
    (b1, "linear alpha=1/3"),
    (p1, "model alpha=1/3"),
    (y1, f"truth {times[VIS_I+1]:%H:%M}"),
    (np.abs(p1-y1), "abs error"),
    (f3, f"input {times[VIS_I+3]:%H:%M}"),
    (b2, "linear alpha=2/3"),
    (p2, "model alpha=2/3"),
    (y2, f"truth {times[VIS_I+2]:%H:%M}"),
    (np.abs(p2-y2), "abs error"),
]
for a, (img, title) in zip(ax.ravel(), items):
    a.imshow(img, cmap="gray_r", vmin=0, vmax=1)
    a.set_title(title)
    a.axis("off")
plt.tight_layout()
fig_path = OUT_DIR / "proxy_window_visual.png"
plt.savefig(fig_path, dpi=140, bbox_inches="tight")
print("saved", fig_path)
plt.show()

## 8. Direct 30-minute endpoint inference: synthesize +10 and +20 minute fog maps

In [ ]:
direct_rows = []
full_preds = {}
for i in range(N - 1):
    print(f"direct pair {i+1}/{N-1}: {times[i]:%H:%M} -> {times[i+1]:%H:%M}")
    p10 = interp_tiled(frames[i], frames[i+1], masks[i], masks[i+1], 1/3)
    p20 = interp_tiled(frames[i], frames[i+1], masks[i], masks[i+1], 2/3)
    t10 = times[i] + (times[i+1] - times[i]) / 3
    t20 = times[i] + 2 * (times[i+1] - times[i]) / 3

    for tag, pred, tt in [("plus_10min", p10, t10), ("plus_20min", p20, t20)]:
        fog_area = int((pred[masks[i] & masks[i+1]] >= THR).sum())
        direct_rows.append({
            "start_time": times[i],
            "pred_time": tt,
            "end_time": times[i+1],
            "target": tag,
            "valid_pixels": int((masks[i] & masks[i+1]).sum()),
            "pred_fog_pixels": fog_area,
            "pred_mean_probability": float(pred[masks[i] & masks[i+1]].mean()),
        })
        png = (np.clip(pred, 0, 1) * 255).astype(np.uint8)
        png_path = OUT_DIR / f"pred_{tt:%Y%m%d_%H%M}_{tag}.png"
        imageio.imwrite(png_path, png)
        if SAVE_FULL_NPZ:
            full_preds[f"{tt:%Y%m%d_%H%M}_{tag}"] = pred.astype(np.float16)

direct = pd.DataFrame(direct_rows)
direct_path = OUT_DIR / "insat_direct_prediction_summary.csv"
direct.to_csv(direct_path, index=False)
print("saved", direct_path)
if SAVE_FULL_NPZ:
    npz_path = OUT_DIR / "insat_direct_predictions_float16.npz"
    np.savez_compressed(npz_path, **full_preds)
    print("saved", npz_path)
display(direct.head())

## 9. Visualize one direct interpolation pair

In [ ]:
PAIR_I = 0
p10 = interp_tiled(frames[PAIR_I], frames[PAIR_I+1], masks[PAIR_I], masks[PAIR_I+1], 1/3)
p20 = interp_tiled(frames[PAIR_I], frames[PAIR_I+1], masks[PAIR_I], masks[PAIR_I+1], 2/3)

fig, ax = plt.subplots(1, 4, figsize=(16, 4.5))
items = [
    (frames[PAIR_I], f"real {times[PAIR_I]:%H:%M}"),
    (p10, "+10 min model"),
    (p20, "+20 min model"),
    (frames[PAIR_I+1], f"real {times[PAIR_I+1]:%H:%M}"),
]
for a, (img, title) in zip(ax, items):
    a.imshow(img, cmap="gray_r", vmin=0, vmax=1)
    a.set_title(title)
    a.axis("off")
plt.tight_layout()
fig_path = OUT_DIR / "direct_pair_visual.png"
plt.savefig(fig_path, dpi=140, bbox_inches="tight")
print("saved", fig_path)
plt.show()

## 10. Output files

After running, download or commit these Kaggle working-directory outputs:

- `insat_proxy_eval_metrics.csv`: baseline vs model proxy metrics.
- `insat_direct_prediction_summary.csv`: synthesized +10/+20 minute timestamps and fog-pixel counts.
- `pred_*.png`: direct synthesized fog probability maps.
- `proxy_window_visual.png`, `direct_pair_visual.png`: quick visual QA figures.
- Optional `insat_direct_predictions_float16.npz` if `SAVE_FULL_NPZ=True`.